In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.ticker import PercentFormatter

# Each model folder holds baseline/ (BBQ before unlearning) and unlearned/ (the run folder).
RESULTS = Path.cwd()

BEFORE, AFTER = "#a3a29b", "#2a78d6"
FORGET, RETAIN = "#2a78d6", "#eb6834"
INK, MUTED, GRID = "#0b0b0b", "#52514e", "#e4e3df"

plt.rcParams.update(
    {
        "figure.dpi": 110,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.edgecolor": MUTED,
        "xtick.color": MUTED,
        "ytick.color": MUTED,
        "axes.grid": True,
        "axes.grid.axis": "y",
        "grid.color": GRID,
        "axes.axisbelow": True,
        "legend.frameon": False,
    }
)

LABELS = {
    "overall_accuracy": "Accuracy (all)",
    "ambig_accuracy": "Accuracy (ambiguous)",
    "disambig_accuracy": "Accuracy (disambiguated)",
    "s_amb": "S_AMB",
    "s_dis": "S_DIS",
}


def read_summary(csv_path):
    row = pd.read_csv(csv_path).set_index("category").loc["OVERALL"]
    return {m: float(row[m]) for m in LABELS}


def plot_eval(eval_csv, baseline_csv, title):
    before, after = read_summary(baseline_csv), read_summary(eval_csv)
    fig, (ax_acc, ax_bias) = plt.subplots(
        1, 2, figsize=(12, 4), gridspec_kw={"width_ratios": [3, 2]}
    )
    for ax, metrics, fmt in [
        (ax_acc, ["overall_accuracy", "ambig_accuracy", "disambig_accuracy"], "{:.1%}"),
        (ax_bias, ["s_amb", "s_dis"], "{:+.3f}"),
    ]:
        x = np.arange(len(metrics))
        w = 0.36
        for offset, values, colour, label in [
            (-w / 2 - 0.01, before, BEFORE, "before"),
            (w / 2 + 0.01, after, AFTER, "after unlearning"),
        ]:
            bars = ax.bar(x + offset, [values[m] for m in metrics], w, color=colour, label=label)
            ax.bar_label(
                bars, labels=[fmt.format(values[m]) for m in metrics], fontsize=8, padding=2
            )
        ax.set_xticks(x, [LABELS[m] for m in metrics])
    ax_acc.set_ylim(0, 1.1)
    ax_acc.yaxis.set_major_formatter(PercentFormatter(1.0))
    ax_acc.set_title("Accuracy", loc="left", color=INK)
    ax_bias.axhline(0, color=INK, lw=0.8)
    lim = max(0.05, *(abs(v[m]) for v in (before, after) for m in ("s_amb", "s_dis"))) * 1.35
    ax_bias.set_ylim(-lim, lim)
    ax_bias.set_title("Bias score", loc="left", color=INK)
    fig.suptitle(title, x=0.01, ha="left", color=INK)
    fig.legend(*ax_acc.get_legend_handles_labels(), loc="upper right", ncol=2)
    fig.tight_layout(rect=(0, 0, 1, 0.93))
    plt.show()


def plot_training(losses_csv, title, window=10):
    losses = pd.read_csv(losses_csv)
    smooth = lambda s: s.rolling(window, min_periods=1).mean()  # noqa: E731
    fig, axes = plt.subplots(1, 3, figsize=(15, 3.6))

    drop = 1 - losses["unlearn_loss"] / losses["unlearn_loss_frozen"]
    axes[0].plot(losses["step"], smooth(drop), color=FORGET, lw=2)
    axes[0].axhline(0, color=INK, lw=0.8)
    axes[0].yaxis.set_major_formatter(PercentFormatter(1.0))
    axes[0].set_title("Forget loss drop, 1 − L/L₀", loc="left", color=INK, fontsize=10)

    for ax, column, name, ref in [
        (axes[1], "cos_u", "cos(a, u)", 0),
        (axes[2], "norm_ratio", "|a| / |a₀|", 1),
    ]:
        ax.plot(
            losses["step"], smooth(losses[f"forget_{column}"]), color=FORGET, lw=2, label="forget"
        )
        ax.plot(
            losses["step"], smooth(losses[f"retain_{column}"]), color=RETAIN, lw=2, label="retain"
        )
        ax.axhline(ref, color=INK, lw=0.8)
        ax.set_title(name, loc="left", color=INK, fontsize=10)
    axes[1].legend(loc="lower right")
    for ax in axes:
        ax.set_xlabel("step")
    fig.suptitle(title, x=0.01, ha="left", color=INK)
    fig.tight_layout()
    plt.show()


def find_one(folder, name):
    """The single `name` file anywhere under `folder` (flat copy or the run folder's bbq/)."""
    matches = sorted(folder.rglob(name))
    if len(matches) != 1:
        raise FileNotFoundError(f"expected one {name} under {folder}, found {len(matches)}")
    return matches[0]


def show_model(model, title):
    folder = RESULTS / model
    unlearned = folder / "unlearned"
    plot_eval(
        find_one(unlearned, "eval_summary.csv"), folder / "baseline" / "eval_summary.csv", title
    )
    plot_training(find_one(unlearned, "losses.csv"), title)

In [ ]:
show_model("lapa-12b", "Lapa 12B")

In [ ]:
show_model("gemma3-12b", "Gemma 3 12B")

In [ ]:
show_model("qwen3-8b", "Qwen3 8B")